# Demo 05 — Regression Models

**Machine Learning with scikit-learn — Session 05**

รันตามสไลด์คาบ 5 — เปิด black box: Linear / Ridge / Lasso / Tree / KNN regression บนข้อมูลราคาบ้านแคลิฟอร์เนีย

## Slide 5 — California housing: 20,640 block groups

y = มูลค่าบ้านมัธยฐานของย่าน (หน่วย ×100k USD) — เราจะใช้โปรโตคอลคาบ 4: split ก่อน + baseline + test ครั้งเดียว

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X, y = fetch_california_housing(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
print(X.shape, "| target mean:", round(y.mean(), 2), "× $100k")

(20640, 8) | target mean: 2.07 × $100k


## Slide 7 — Round 1: baseline vs linear

In [2]:
def report(name, model):
    m = make_pipeline(StandardScaler(), model).fit(Xtr, ytr)
    yp = m.predict(Xte)
    print(f"{name:26s} MAE={mean_absolute_error(yte, yp):7.3f}  R2={r2_score(yte, yp):6.3f}")

report("Dummy(median)", DummyRegressor(strategy="median"))
report("LinearRegression", LinearRegression())

Dummy(median)              MAE=  0.874  R2=-0.050
LinearRegression           MAE=  0.533  R2= 0.576


Dummy ได้ R² **ติดลบ** (-0.05) = แย่กว่าทายค่าเฉลี่ยธรรมดา — นี่คือพื้นบ้านของโมเดลทุกตัว

## Slide 6 — Reading the model's mind (coefficients)

In [3]:
import pandas as pd
lr = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr, ytr)
names = fetch_california_housing().feature_names
pd.Series(lr[-1].coef_, index=names).round(2).sort_values(key=abs, ascending=False)

Latitude     -0.90
Longitude    -0.87
MedInc        0.85
AveBedrms     0.34
AveRooms     -0.29
HouseAge      0.12
AveOccup     -0.04
Population   -0.00
dtype: float64

- `MedInc +0.85σ` รายได้สูง → บ้านแพง
- `Latitude/Longitude` หนักมาก — แคลิฟอร์เนียคือเรื่องของภูมิศาสตร์ (ชายฝั่ง!)
- intercept ≈ 2.07 = ราคาเฉลี่ย ($207k) เมื่อทุกฟีเจอร์อยู่ที่ค่าเฉลี่ย
- ระวัง: นี่คือ *คำอธิบาย* ไม่ใช่ *ตัวกำหนดเหตุผล* (correlation ≠ causation)

## Slide 10 — Regularization: Ridge & Lasso

In [4]:
report("Ridge(alpha=1)", Ridge(alpha=1.0))
report("Lasso(alpha=0.1)", Lasso(alpha=0.1))

Ridge(alpha=1)             MAE=  0.533  R2= 0.576
Lasso(alpha=0.1)           MAE=  0.622  R2= 0.481


- Ridge ≈ Linear เปี๊ยบ (0.576) — ข้อมูลนี้มีแค่ 8 ฟีเจอร์สะอาด ๆ ไม่มีอะไรให้ "งด"
- Lasso(0.1) แย่ลง (0.481) — ลงโทษหนักเกินไป: **alpha คือ hyperparameter ที่ต้อง tune** (คาบ 7)

regularization จะมีค่าตอบแทนเมื่อฟีเจอร์เยอะ/สัมพันธ์กันเอง/ใช้ polynomial features

## Slide 11 — Trees: prediction = average of the room

tree ที่ปล่อยอิสระ vs จำกัด max_depth=8 — นี่คือบทเรียน overfitting จากคาบ 4 อีกครั้ง

In [5]:
free_tree = DecisionTreeRegressor(random_state=42).fit(Xtr, ytr)
print(f"free tree : train R2={free_tree.score(Xtr, ytr):.3f}  test R2={free_tree.score(Xte, yte):.3f}  ← ท่องจำหมดแล้ว")

report("Tree(depth=8)", DecisionTreeRegressor(max_depth=8, random_state=42))

free tree : train R2=1.000  test R2=0.623  ← ท่องจำหมดแล้ว


Tree(depth=8)              MAE=  0.448  R2= 0.678


## Slide 12 — KNN regression: neighbors average their values

โมเดลเดียวกับคาบ 1 แต่ผลลัพธ์เป็น "ค่าเฉลี่ยของเพื่อนบ้าน" แทน "เสียงโหวต"

In [6]:
report("KNN(k=10)", KNeighborsRegressor(n_neighbors=10))

KNN(k=10)                  MAE=  0.440  R2= 0.678


## Slide 13 — Scoreboard + human translation

In [7]:
print("พื้นบ้าน (Dummy) MAE 0.874 → KNN 0.440")
print("0.440 × $100k = พลาดเฉลี่ย ~$44,000 ต่อการทำนาย 1 ครั้ง")

พื้นบ้าน (Dummy) MAE 0.874 → KNN 0.440
0.440 × $100k = พลาดเฉลี่ย ~$44,000 ต่อการทำนาย 1 ครั้ง


ตารางสุดท้าย (จากสไลด์): Dummy −0.05 · Linear 0.576 · Ridge 0.576 · Lasso(0.1) 0.481 · Tree ฟรี 0.623 (train 1.000!) · Tree(8) 0.678 · KNN 0.678

โมเดลเลขเดียวกัน แต่ "เรื่องเล่า" ต่างกัน: Linear อ่านสัมประสิทธิ์ได้ / Tree ให้กฎ / KNN ให้ logic ย่านเดียวกันราคาใกล้กัน —
เลือกตามสิ่งที่ stakeholder ต้องการ: **prediction หรือ explanation**

---
✅ **จบ Demo 05** — **Lab 05**: `excercises/05.regression_exercise.ipynb`